# 🐾 PawMatch — Pet Breed Identifier
### Notebook Demo

This notebook walks through the **full pipeline** step by step:
1. Load CLIP model
2. Extract image embeddings
3. Zero-shot breed classification
4. Cosine similarity comparison
5. Same / Different verdict

---

## 0. Setup

In [ ]:
# Install dependencies if running in Colab
import sys
# !{sys.executable} -m pip install transformers torch Pillow numpy streamlit

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from breed_identifier import BreedIdentifier, ALL_BREEDS, DOG_BREEDS, CAT_BREEDS

print(f'PyTorch: {torch.__version__}')
print(f'Device : {"CUDA" if torch.cuda.is_available() else "CPU"}')
print(f'Breeds : {len(ALL_BREEDS)} ({len(DOG_BREEDS)} dogs + {len(CAT_BREEDS)} cats)')

## 1. Load Model

In [ ]:
# First run downloads ~340 MB CLIP weights; subsequent runs use local cache
identifier = BreedIdentifier()
print('Model loaded')

## 2. Understand the Embedding Space

In [ ]:
emb = identifier._breed_text_embeddings
print(f'Text embedding matrix shape: {emb.shape}')
print(f' → {emb.shape[0]} breeds × {emb.shape[1]}-dim embeddings')
print(f'Embeddings are L2-normalised: {torch.allclose(emb.norm(dim=-1), torch.ones(emb.shape[0]), atol=1e-5)}')

## 3. Single Image — Breed Identification

In [ ]:
def show_prediction(img_path: str, ax=None):
    """Display an image with its top-3 breed predictions."""
    img = Image.open(img_path)
    pred = identifier.identify_breed(img)

    if ax is None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        ax_img, ax_bar = axes
    else:
        ax_img, ax_bar = ax

    ax_img.imshow(img)
    ax_img.axis('off')
    icon = '🐕' if pred.pet_type == 'dog' else '🐈'
    ax_img.set_title(f'{icon} {pred.breed}\n{pred.confidence*100:.1f}% confidence',
                     fontsize=11, fontweight='bold')

    breeds = [b for b, _ in pred.top_3]
    scores = [s * 100 for _, s in pred.top_3]
    colors = ['#2dbd6e', '#e5a00d', '#4a9eff']
    ax_bar.barh(breeds[::-1], scores[::-1], color=colors[::-1], height=0.5)
    ax_bar.set_xlim(0, max(scores) * 1.25)
    ax_bar.set_xlabel('Confidence (%)')
    ax_bar.set_title('Top-3 Breed Predictions')
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)

    plt.tight_layout()
    return pred

# Replace with your own image path:
# pred = show_prediction('examples/golden_1.jpg')
# plt.show()

## 4. Pair Comparison — Same / Different Breed

In [ ]:
def compare_and_plot(path1: str, path2: str):
    """Compare two images and visualise the result."""
    img1, img2 = Image.open(path1), Image.open(path2)
    result = identifier.compare(img1, img2)

    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 5, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    axv = fig.add_subplot(gs[0, 2])
    axb1 = fig.add_subplot(gs[0, 3])
    axb2 = fig.add_subplot(gs[0, 4])

    for ax, img, pred, label in [
        (ax1, img1, result.image1_prediction, 'Image 1'),
        (ax2, img2, result.image2_prediction, 'Image 2'),
    ]:
        ax.imshow(img)
        ax.axis('off')
        icon = '🐕' if pred.pet_type == 'dog' else '🐈'
        ax.set_title(f'{label}\n{icon} {pred.breed}', fontsize=9, fontweight='bold')

    color = '#2dbd6e' if result.same_breed else '#e05050'
    verdict_text = 'SAME\nBREED' if result.same_breed else 'DIFFERENT\nBREEDS'
    icon_text = '✅' if result.same_breed else '❌'

    axv.set_facecolor('#0d1b2a')
    axv.text(0.5, 0.60, icon_text, ha='center', va='center',
             transform=axv.transAxes, fontsize=28)
    axv.text(0.5, 0.38, verdict_text, ha='center', va='center',
             transform=axv.transAxes, fontsize=13, fontweight='bold',
             color=color, linespacing=1.3)
    axv.text(0.5, 0.14,
             f'Confidence: {result.confidence*100:.1f}%\nSimilarity: {result.similarity_score:.3f}',
             ha='center', va='center', transform=axv.transAxes,
             fontsize=8.5, color='#a0b4c8', linespacing=1.5)
    axv.set_xticks([]); axv.set_yticks([])

    for ax, pred in [(axb1, result.image1_prediction), (axb2, result.image2_prediction)]:
        breeds = [b[:18] for b, _ in pred.top_3]
        scores = [s * 100 for _, s in pred.top_3]
        ax.barh(breeds[::-1], scores[::-1],
                color=['#2dbd6e', '#e5a00d', '#4a9eff'][::-1], height=0.5)
        ax.set_xlim(0, max(scores) * 1.3)
        ax.set_xlabel('Confidence (%)', fontsize=8)
        ax.set_title('Top-3 Matches', fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('PawMatch — Breed Comparison', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    print('\n' + '─'*55)
    print(result.verdict)
    print('─'*55)
    return result

# Replace with your own image paths:
# result = compare_and_plot('examples/golden_1.jpg', 'examples/golden_2.jpg')

## 5. Similarity Score Distribution (concept)

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4))

np.random.seed(42)
same_breed_sim = np.clip(np.random.normal(0.88, 0.05, 500), 0.5, 1.0)
diff_breed_sim = np.clip(np.random.normal(0.72, 0.07, 500), 0.4, 1.0)

ax.hist(diff_breed_sim, bins=40, alpha=0.6, color='#e05050', label='Different Breed pairs')
ax.hist(same_breed_sim, bins=40, alpha=0.6, color='#2dbd6e', label='Same Breed pairs')
ax.axvline(0.82, color='#e5a00d', linewidth=2, linestyle='--', label='Decision threshold (0.82)')

ax.set_xlabel('Cosine Similarity', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Expected Similarity Distribution — Same vs Different Breed Pairs\n(illustrative; calibrate on your dataset)', fontsize=10)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()